In [79]:
from dotenv import load_dotenv
from pathlib import Path
import os

env_path = Path.cwd().parent.parent / ".env"
load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")


In [ ]:
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command
from rich import print
import uuid

In [81]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage, SystemMessage
from typing import Annotated
from operator import add

In [82]:
model = ChatOpenAI(
    model ='gpt-4.1-mini',
    api_key = api_key,
    temperature=0
)

In [83]:
class State(BaseModel):
    messages: Annotated[list[BaseMessage], add] = Field(default_factory=list)

In [84]:
@tool
def add(numbers: list[float]) -> float:
    """THis tool sum up the value of all the arguments passed into it."""
    total = 0
    # Loop through the tuple of arguments
    for num in numbers:
        total += num
    return total

In [85]:
@tool
def substract(num1:float, num2:float)->float:
    """ This tool is used to find the difference between the two numbers"""
    return num1-num2

In [86]:
@tool
def multiply(numbers: list[float]) -> float:
    """ This tool is used to multiply the values of all the arguments passed into it"""
    product=1
    # Loop through multiple arguments
    for num in numbers:
        product = product*num
    return product

In [88]:
@tool
def divide(num1:float, num2:float)->float:
    """THis tool is used to divide the argument passed into it."""
    try:
        if(num2!=0):
            return num1/num2
    except ZeroDivisionError:
        print("Can not divide by 0")

In [89]:
tools = [add, substract, multiply, divide]
tool_by_name = {tool.name: tool for tool in tools}
model_with_tools = model.bind_tools(tools)

In [97]:
def execute_agent(state:State):

    messages = state.messages

    if not messages:
        messages.append(
            SystemMessage(
                content="You are a helpful calculator assistant. Use tools to answer arithmetic questions."
            )
        )

        messages.append(
            HumanMessage(content="solve 12 ÷ 2 × 3 - 5")
        )
        
    while True:
        response = model_with_tools.invoke(messages)

        if not response.tool_calls:
            return response.content

        messages.append(response)
        for tool_call in response.tool_calls:
            selected_tool = tool_by_name[tool_call["name"]]
            tool_result = selected_tool.invoke(tool_call["args"])
            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"],
                )
            )
                

In [98]:
state = State()
result = execute_agent(state)
print(result)

The result of 12 ÷ 2 × 3 - 5 is 1.